# 09 — Publication tables and double-column figures

Set `runtime.mode` to `demo` for notebooks 07–08 synthetic outputs, or `public-results` to render figures from released aggregate files in `data/public`. Public-results mode never opens paper/review samples or `data/work`; missing real JSD bins explicitly skip the histogram. The separately released JSD distribution image is not regenerated from the released aggregate data. The public-results branch cannot refit models; it renders supplied combined-model coefficient data. Outputs remain separated under `results/demo` or `results/regenerated`.

SOURCE_LINEAGE_LITERAL: `make_publication_figures_tables.ipynb` cells 0–6 and 8–10; `main_analysis_strict_sample.ipynb` cell 9 supplies figure-data provenance. The source publication formatting (stars, p strings, HC3 combined model, RdBu_r heatmap) is retained in demo mode.


In [ ]:
from __future__ import annotations

import math
import re
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

try:
    import mpmath as mp
except Exception:  # mpmath is optional
    mp = None
from matplotlib.colors import TwoSlopeNorm
from pathlib import Path
import sys
import os
import yaml

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "project_paths.py").exists():
    ROOT = ROOT.parent
if not (ROOT / "src" / "project_paths.py").exists():
    raise FileNotFoundError("Run from the release root or its notebooks directory.")
sys.path.insert(0, str(ROOT))
from src.project_paths import project_path

with (ROOT / "config" / "config.example.yaml").open(encoding="utf-8") as fh:
    CONFIG = yaml.safe_load(fh)
MODE = os.environ.get("NC_RUNTIME_MODE", CONFIG.get("runtime", {}).get("mode", "demo"))
if MODE not in ('demo', 'public-results'):
    raise ValueError("This notebook requires runtime.mode in: demo, public-results")
ANALYSIS = CONFIG["analysis"]
if int(ANALYSIS["formal_country_min_papers"]) != 500:
    raise ValueError("The formal country threshold must remain 500.")
if int(ANALYSIS["large_team_cutoff"]) != 50:
    raise ValueError("The formal large-team cutoff must remain 50.")
if ANALYSIS["country_reference"] != "US" or ANALYSIS["gender_reference"] != "male":
    raise ValueError("Formal model references must remain US and male.")
ANALYSIS_ASPECT_ORDER = ['Methodology', 'Results', 'Experiments and Evaluation', 'Data and Datasets', 'Figures and Tables', 'Writing, Language and Terminology', 'Theory and Models', 'Comparison and Context', 'Paper Assessment', 'Novelty and Impact', 'Validity, Rigor and Quality', 'Limitations']
JSD_COL = "mean_pairwise_jsd"
COUNTRY_COL = "first_author_country_group_n500_strict"
GENDER_COL = "first_author_gender_confident"
COUNTRY_REFERENCE = ANALYSIS["country_reference"]
GENDER_REFERENCE = ANALYSIS["gender_reference"]
PUBLIC_DATA_DIR = project_path(CONFIG["paths"]["public_results_data_dir"])
PUBLIC_FIGURE_DATA = PUBLIC_DATA_DIR / "figure_data"
PUBLIC_TABLE_DATA = PUBLIC_DATA_DIR / "publication_tables"
if MODE == "demo":
    SAMPLE_DIR = project_path("results/demo/strict/samples")
    PAPER_MAIN_PATH = SAMPLE_DIR / "main_strict_paper_jsd_no_large50.parquet"
    MAIN_TABLE_DIR = project_path("results/demo/strict/tables")
    ROBUST1_TABLE_DIR = project_path("results/demo/robustness/first_batch/tables")
    OUT_TABLE_DIR = project_path("results/demo/publication")
    OUT_FIG_DIR = project_path("results/demo/figures/double_column")
    IN_DIR = MAIN_TABLE_DIR
else:
    # Aggregate-only public-results mode: no sample or work-data path exists here.
    OUT_TABLE_DIR = project_path("results/regenerated/publication")
    OUT_FIG_DIR = project_path("results/regenerated/figures/double_column")
    IN_DIR = PUBLIC_FIGURE_DATA
OUT_TABLE_DIR.mkdir(parents=True, exist_ok=True)
OUT_FIG_DIR.mkdir(parents=True, exist_ok=True)
def save_csv(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"Saved: {path} | rows={len(df)}, cols={len(df.columns)}")


def safe_log1p(x: pd.Series) -> pd.Series:
    return np.log1p(pd.to_numeric(x, errors="coerce").clip(lower=0))


def coerce_for_statsmodels(
    df: pd.DataFrame,
    numeric_cols: list[str] | None = None,
    categorical_cols: list[str] | None = None,
    group_col: Optional[str] = None,
) -> pd.DataFrame:
    """Convert pandas nullable dtypes to NumPy/Python dtypes before statsmodels formulas."""
    out = df.copy()
    numeric_cols = numeric_cols or []
    categorical_cols = categorical_cols or []

    for col in numeric_cols:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce").astype("float64")

    for col in categorical_cols:
        if col in out.columns:
            out[col] = out[col].astype("object").where(out[col].notna(), "Unknown").astype(str)

    if group_col is not None and group_col in out.columns:
        out[group_col] = out[group_col].astype("object").astype(str)

    for col in out.columns:
        if pd.api.types.is_extension_array_dtype(out[col].dtype):
            if pd.api.types.is_numeric_dtype(out[col].dtype):
                out[col] = pd.to_numeric(out[col], errors="coerce").astype("float64")
            elif pd.api.types.is_bool_dtype(out[col].dtype):
                out[col] = out[col].astype("float64")
            else:
                out[col] = out[col].astype("object").where(out[col].notna(), "Unknown").astype(str)

    return out


def p_stars(p: float | int | None) -> str:
    if p is None or pd.isna(p):
        return ""
    p = float(p)
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return ""


def format_p(p: float | int | None) -> str:
    """Publication-style p-value formatting: never write p = 0.000."""
    if p is None or pd.isna(p):
        return ""
    p = float(p)
    if p < 0.001:
        return "<0.001"
    return f"{p:.3f}"


def format_num(x: float | int | None, digits: int = 3) -> str:
    if x is None or pd.isna(x):
        return ""
    return f"{float(x):.{digits}f}"


def chi_square_log10_p(chi2: float, dof: int, dps: int = 100) -> Optional[float]:
    """
    Return log10 survival probability for chi-square test using mpmath.
    Useful when scipy/pandas output underflows to p=0.
    """
    if mp is None:
        return None
    try:
        mp.mp.dps = dps
        a = mp.mpf(dof) / 2
        x = mp.mpf(chi2) / 2
        # upper regularized gamma Q(a, x) = gammainc(a, x, inf) / gamma(a)
        p = mp.gammainc(a, x, mp.inf) / mp.gamma(a)
        if p <= 0:
            return None
        return float(mp.log10(p))
    except Exception:
        return None


def clean_term_label(term: str) -> str:
    """Convert statsmodels categorical term into readable label."""
    m = re.search(r"\[T\.(.*?)\]", str(term))
    if m:
        return m.group(1)
    return str(term)


def extract_model_rows(model, model_name: str, outcome: str) -> pd.DataFrame:
    rows = []
    conf = model.conf_int()
    for term, est in model.params.items():
        if term == "Intercept":
            continue
        lo, hi = conf.loc[term]
        rows.append({
            "model": model_name,
            "outcome": outcome,
            "term": term,
            "label": clean_term_label(term),
            "estimate": float(est),
            "std_error": float(model.bse[term]),
            "conf_low": float(lo),
            "conf_high": float(hi),
            "p_value": float(model.pvalues[term]),
            "nobs": int(model.nobs),
            "r_squared": float(getattr(model, "rsquared", np.nan)),
        })
    out = pd.DataFrame(rows)
    if len(out) > 0:
        out["p_adj_bh"] = multipletests(out["p_value"].fillna(1), method="fdr_bh")[1]
    return out


def format_regression_table(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if out.empty:
        return out
    out["estimate_fmt"] = out["estimate"].map(lambda x: format_num(x, 4))
    out["std_error_fmt"] = out["std_error"].map(lambda x: format_num(x, 4))
    out["ci_95_fmt"] = out.apply(lambda r: f"[{format_num(r['conf_low'], 4)}, {format_num(r['conf_high'], 4)}]", axis=1)
    out["p_fmt"] = out["p_value"].map(format_p)
    out["p_adj_bh_fmt"] = out["p_adj_bh"].map(format_p) if "p_adj_bh" in out.columns else ""
    out["sig"] = out["p_value"].map(p_stars)
    out["sig_fdr"] = out["p_adj_bh"].map(p_stars) if "p_adj_bh" in out.columns else ""
    return out


## Demo publication tables and coefficient figures

In [ ]:
if MODE == "demo":
    # =========================================================
    # 2. Load strict main sample
    # =========================================================

    if not PAPER_MAIN_PATH.exists():
        raise FileNotFoundError(f"Cannot find {PAPER_MAIN_PATH}. Run strict main analysis first.")

    paper = pd.read_parquet(PAPER_MAIN_PATH)

    # Ensure needed transformed variables exist.
    if "log_aspect_total_count" not in paper.columns and "aspect_total_count" in paper.columns:
        paper["log_aspect_total_count"] = safe_log1p(paper["aspect_total_count"])
    if "log_n_authors" not in paper.columns and "n_authors" in paper.columns:
        paper["log_n_authors"] = safe_log1p(paper["n_authors"])


    # =========================================================
    # 3. Figure: JSD distribution
    # =========================================================

    jsd = pd.to_numeric(paper[JSD_COL], errors="coerce").dropna()

    fig, ax = plt.subplots(figsize=(6.5, 4.2), dpi=300)
    ax.hist(jsd, bins=40, density=True, alpha=0.85, edgecolor="white", linewidth=0.4)
    ax.axvline(jsd.mean(), linestyle="--", linewidth=1.2, label=f"Mean = {jsd.mean():.3f}")
    ax.axvline(jsd.median(), linestyle=":", linewidth=1.2, label=f"Median = {jsd.median():.3f}")
    ax.set_xlabel("Mean pairwise JSD")
    ax.set_ylabel("Density")
    ax.legend(frameon=False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    fig.tight_layout()
    fig.savefig(OUT_FIG_DIR / "fig_jsd_distribution.png", bbox_inches="tight")
    fig.savefig(OUT_FIG_DIR / "fig_jsd_distribution.pdf", bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {OUT_FIG_DIR / 'fig_jsd_distribution.png'}")

    # =========================================================
    # 4. Fit combined paper-level JSD model and plot coefficients
    # =========================================================

    needed = [
        "paper_code",
        JSD_COL,
        COUNTRY_COL,
        GENDER_COL,
        "publication_year",
        "subject_primary",
        "n_reviews_valid_for_jsd",
        "log_aspect_total_count",
        "log_n_authors",
    ]
    missing = [c for c in needed if c not in paper.columns]
    if missing:
        raise KeyError(f"Missing columns for combined JSD model: {missing}")

    model_df = paper[needed].copy()
    model_df = model_df[
        model_df[COUNTRY_COL].notna()
        & (model_df[COUNTRY_COL].astype(str) != "Unknown")
        & model_df[GENDER_COL].isin(["male", "female"])
    ].dropna().copy()

    model_df = coerce_for_statsmodels(
        model_df,
        numeric_cols=[JSD_COL, "publication_year", "n_reviews_valid_for_jsd", "log_aspect_total_count", "log_n_authors"],
        categorical_cols=[COUNTRY_COL, GENDER_COL, "subject_primary"],
        group_col="paper_code",
    )

    formula = (
        f"{JSD_COL} ~ "
        f"C({COUNTRY_COL}, Treatment(reference='{COUNTRY_REFERENCE}')) + "
        f"C({GENDER_COL}, Treatment(reference='{GENDER_REFERENCE}')) + "
        "publication_year + C(subject_primary) + n_reviews_valid_for_jsd + "
        "log_aspect_total_count + log_n_authors"
    )

    combined_model = smf.ols(formula, data=model_df).fit(cov_type="HC3")
    combined_rows = extract_model_rows(combined_model, "country_gender_combined", JSD_COL)
    combined_rows["n_papers"] = model_df["paper_code"].nunique()
    save_csv(combined_rows, OUT_TABLE_DIR / "model_paper_jsd_ols__country_gender_combined_main.csv")
    save_csv(format_regression_table(combined_rows), OUT_TABLE_DIR / "model_paper_jsd_ols__country_gender_combined_main_formatted.csv")

    # Sample count table for the combined model.
    combined_sample_count = pd.DataFrame([{
        "sample": "paper_jsd_country_gender_combined",
        "n_papers": model_df["paper_code"].nunique(),
        "n_reviews": int(pd.to_numeric(model_df.get("n_reviews_valid_for_jsd", pd.Series(dtype=float)), errors="coerce").sum()),
        "nobs": int(combined_model.nobs),
    }])
    save_csv(combined_sample_count, OUT_TABLE_DIR / "sample_count__paper_jsd_country_gender_combined.csv")

    # Country coefficients from the combined model.
    country_pat = f"C({COUNTRY_COL}"
    gender_pat = f"C({GENDER_COL}"
    country_coef = combined_rows[combined_rows["term"].str.contains(re.escape(country_pat), regex=True, na=False)].copy()
    gender_coef = combined_rows[combined_rows["term"].str.contains(re.escape(gender_pat), regex=True, na=False)].copy()

    # Plot country coefficients.
    if not country_coef.empty:
        country_coef = country_coef.sort_values("estimate")
        y = np.arange(len(country_coef))
        fig, ax = plt.subplots(figsize=(6.8, max(4.2, 0.35 * len(country_coef))), dpi=300)
        ax.errorbar(
            country_coef["estimate"],
            y,
            xerr=[country_coef["estimate"] - country_coef["conf_low"], country_coef["conf_high"] - country_coef["estimate"]],
            fmt="o",
            capsize=3,
            linewidth=1,
            markersize=4,
        )
        ax.axvline(0, linestyle="--", linewidth=1)
        ax.set_yticks(y)
        ax.set_yticklabels(country_coef["label"])
        ax.set_xlabel("Coefficient on mean pairwise JSD")
        ax.set_ylabel("Country/region (reference: US)")
        for yi, est in zip(y, country_coef["estimate"]):
            ax.text(est, yi, f" {est:.4f}", va="center", fontsize=8)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        fig.tight_layout()
        fig.savefig(OUT_FIG_DIR / "fig_jsd_combined_country_coefficients.png", bbox_inches="tight")
        fig.savefig(OUT_FIG_DIR / "fig_jsd_combined_country_coefficients.pdf", bbox_inches="tight")
        plt.close(fig)
        print(f"Saved: {OUT_FIG_DIR / 'fig_jsd_combined_country_coefficients.png'}")

    # Plot gender coefficient.
    if not gender_coef.empty:
        gender_coef = gender_coef.copy()
        gender_coef["label"] = gender_coef["label"].replace({"female": "Female vs male"})
        y = np.arange(len(gender_coef))
        fig, ax = plt.subplots(figsize=(5.5, 2.0), dpi=300)
        ax.errorbar(
            gender_coef["estimate"],
            y,
            xerr=[gender_coef["estimate"] - gender_coef["conf_low"], gender_coef["conf_high"] - gender_coef["estimate"]],
            fmt="o",
            capsize=3,
            linewidth=1,
            markersize=4,
        )
        ax.axvline(0, linestyle="--", linewidth=1)
        ax.set_yticks(y)
        ax.set_yticklabels(gender_coef["label"])
        ax.set_xlabel("Coefficient on mean pairwise JSD")
        ax.set_ylabel("Gender")
        for yi, est in zip(y, gender_coef["estimate"]):
            ax.text(est, yi, f" {est:.4f}", va="center", fontsize=8)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        fig.tight_layout()
        fig.savefig(OUT_FIG_DIR / "fig_jsd_combined_gender_coefficients.png", bbox_inches="tight")
        fig.savefig(OUT_FIG_DIR / "fig_jsd_combined_gender_coefficients.pdf", bbox_inches="tight")
        plt.close(fig)
        print(f"Saved: {OUT_FIG_DIR / 'fig_jsd_combined_gender_coefficients.png'}")


    # =========================================================
    # 5. OLS ratio vs Poisson offset consistency summary table
    # =========================================================

    def build_ols_poisson_summary() -> pd.DataFrame:
        rows = []
        for prefix, label in [
            ("country", "Country model"),
            ("gender", "Gender model"),
            ("country_gender", "Country + gender model"),
        ]:
            summary_path = ROBUST1_TABLE_DIR / f"compare_review_ols_ratio_vs_poisson_offset__{prefix}__summary.csv"
            detail_path = ROBUST1_TABLE_DIR / f"compare_review_ols_ratio_vs_poisson_offset__{prefix}.csv"

            if summary_path.exists():
                s = pd.read_csv(summary_path)
                if len(s) > 0:
                    r = s.iloc[0].to_dict()
                    rows.append({
                        "model": label,
                        "n_terms_compared": r.get("n_terms_compared", np.nan),
                        "same_direction_n": r.get("same_direction_n", np.nan),
                        "same_direction_share": r.get("same_direction_share", np.nan),
                        "both_p_lt_05_n": r.get("both_p_lt_05_n", np.nan),
                        "both_p_lt_05_share": r.get("both_p_lt_05_share", np.nan),
                        "both_fdr_lt_05_n": r.get("both_fdr_lt_05_n", np.nan),
                        "both_fdr_lt_05_share": r.get("both_fdr_lt_05_share", np.nan),
                    })
                    continue

            if detail_path.exists():
                d = pd.read_csv(detail_path)
                rows.append({
                    "model": label,
                    "n_terms_compared": len(d),
                    "same_direction_n": int(d["same_direction"].sum()) if "same_direction" in d else np.nan,
                    "same_direction_share": float(d["same_direction"].mean()) if "same_direction" in d else np.nan,
                    "both_p_lt_05_n": int(d["both_p_lt_05"].sum()) if "both_p_lt_05" in d else np.nan,
                    "both_p_lt_05_share": float(d["both_p_lt_05"].mean()) if "both_p_lt_05" in d else np.nan,
                    "both_fdr_lt_05_n": int(d["both_fdr_lt_05"].sum()) if "both_fdr_lt_05" in d else np.nan,
                    "both_fdr_lt_05_share": float(d["both_fdr_lt_05"].mean()) if "both_fdr_lt_05" in d else np.nan,
                })

        return pd.DataFrame(rows)

    consistency = build_ols_poisson_summary()
    save_csv(consistency, OUT_TABLE_DIR / "table_ols_poisson_consistency_summary.csv")

    consistency_fmt = consistency.copy()
    for col in ["same_direction_share", "both_p_lt_05_share", "both_fdr_lt_05_share"]:
        if col in consistency_fmt.columns:
            consistency_fmt[col] = consistency_fmt[col].map(lambda x: "" if pd.isna(x) else f"{100 * float(x):.2f}%")
    for col in ["n_terms_compared", "same_direction_n", "both_p_lt_05_n", "both_fdr_lt_05_n"]:
        if col in consistency_fmt.columns:
            consistency_fmt[col] = consistency_fmt[col].map(lambda x: "" if pd.isna(x) else f"{int(x)}")
    save_csv(consistency_fmt, OUT_TABLE_DIR / "table_ols_poisson_consistency_summary_formatted.csv")

    # =========================================================
    # 6. Formatted chi-square test table with p strings and stars
    # =========================================================

    chi_files = []
    for folder in [MAIN_TABLE_DIR, ROBUST1_TABLE_DIR]:
        if folder.exists():
            chi_files.extend(sorted(folder.glob("test_chi_square_aspect*.csv")))

    chi_rows = []
    for fp in chi_files:
        try:
            df = pd.read_csv(fp)
        except Exception:
            continue
        if df.empty:
            continue
        for _, r in df.iterrows():
            chi2_val = r.get("chi2", np.nan)
            dof = r.get("dof", np.nan)
            p_val = r.get("p_value", np.nan)
            log10p = None
            if pd.notna(chi2_val) and pd.notna(dof):
                log10p = chi_square_log10_p(float(chi2_val), int(dof))
            chi_rows.append({
                "source_file": fp.name,
                "group_col": r.get("group_col", ""),
                "n_groups": r.get("n_groups", ""),
                "n_aspects": r.get("n_aspects", ""),
                "chi2": chi2_val,
                "dof": dof,
                "p_value_raw": p_val,
                "p_log10": log10p,
                "cramers_v": r.get("cramers_v", np.nan),
                "chi2_fmt": format_num(chi2_val, 2),
                "p_fmt": format_p(p_val),
                "sig": p_stars(p_val),
                "cramers_v_fmt": format_num(r.get("cramers_v", np.nan), 3),
            })

    chi_table = pd.DataFrame(chi_rows)
    save_csv(chi_table, OUT_TABLE_DIR / "table_chi_square_tests_formatted.csv")

    print("\nDone. Main outputs:")
    print(f"- {OUT_FIG_DIR / 'fig_jsd_distribution.png'}")
    print(f"- {OUT_FIG_DIR / 'fig_jsd_combined_country_coefficients.png'}")
    print(f"- {OUT_FIG_DIR / 'fig_jsd_combined_gender_coefficients.png'}")
    print(f"- {OUT_TABLE_DIR / 'table_ols_poisson_consistency_summary_formatted.csv'}")
    print(f"- {OUT_TABLE_DIR / 'table_chi_square_tests_formatted.csv'}")


## Approved aggregate-only public rendering

In [ ]:
def read_public_aggregate(name, required_columns):
    allowed = {
        "figure_data_country_aspect_deviation_pp.csv",
        "figure_data_gender_aspect_deviation_pp.csv",
        "figure_data_jsd_country_coefficients.csv",
        "figure_data_jsd_gender_coefficients.csv",
        "descriptive_per_aspect_disagreement_main_strict.csv",
        "jsd_distribution_bins.csv",
    }
    if name not in allowed:
        raise ValueError(f"Not an approved aggregate input: {name}")
    path = PUBLIC_FIGURE_DATA / name
    if not path.is_file():
        return None
    frame = pd.read_csv(path)
    forbidden = {"paper_code", "review_id", "review_text", "author_name", "paper_title"}
    if forbidden.intersection(frame.columns):
        raise ValueError(f"Row-level identifiers are forbidden in {path}")
    missing = set(required_columns).difference(frame.columns)
    if missing:
        raise ValueError(f"Missing aggregate columns in {path}: {sorted(missing)}")
    aspect_columns = set(ANALYSIS_ASPECT_ORDER)
    allowed_columns = {
        "figure_data_country_aspect_deviation_pp.csv": {"group"} | aspect_columns,
        "figure_data_gender_aspect_deviation_pp.csv": {"group"} | aspect_columns,
        "figure_data_jsd_country_coefficients.csv": {"aspect_set", "dependent_var", "predictor_type", "term", "label", "estimate", "conf_low", "conf_high", "p_value", "p_adj_bh", "n_papers", "r_squared", "adj_r_squared", "model"},
        "figure_data_jsd_gender_coefficients.csv": {"aspect_set", "dependent_var", "predictor_type", "term", "label", "estimate", "conf_low", "conf_high", "p_value", "p_adj_bh", "n_papers", "r_squared", "adj_r_squared", "model"},
        "descriptive_per_aspect_disagreement_main_strict.csv": {"aspect_class", "aspect_label", "n_papers", "mean_mapd", "median_mapd", "sd_mapd", "q25_mapd", "q75_mapd"},
        "jsd_distribution_bins.csv": {"bin_left", "bin_right", "count"},
    }
    extra = set(frame.columns).difference(allowed_columns[name])
    if extra:
        raise ValueError(f"Unexpected columns in approved aggregate {path}: {sorted(extra)}")
    return frame

if MODE == "public-results":
    # This branch only reads the explicit aggregate allowlist above.
    bins = read_public_aggregate("jsd_distribution_bins.csv", ["bin_left", "bin_right", "count"])
    if bins is None:
        print(f"SKIP fig_jsd_distribution: {PUBLIC_FIGURE_DATA / 'jsd_distribution_bins.csv'} is absent; no microdata fallback.")
    else:
        bins = bins.apply(pd.to_numeric, errors="raise")
        if bins.empty or (bins["bin_right"] <= bins["bin_left"]).any() or (bins["count"] < 0).any():
            raise ValueError("Invalid public JSD histogram bins")
        fig, ax = plt.subplots(figsize=(6.5, 4.2), dpi=300)
        width = bins["bin_right"] - bins["bin_left"]
        density = bins["count"] / ((bins["count"] * width).sum())
        ax.bar(bins["bin_left"], density, width=width, align="edge", alpha=0.85, edgecolor="white", linewidth=0.4)
        ax.set_xlabel("Mean pairwise JSD")
        ax.set_ylabel("Density")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        fig.tight_layout()
        for suffix in ("png", "pdf"):
            fig.savefig(OUT_FIG_DIR / f"fig_jsd_distribution.{suffix}", bbox_inches="tight")
        plt.close(fig)
    # Already aggregated publication tables are formatting inputs, never model inputs.
    table_allowlist = {
        "model_paper_jsd_ols__country_gender_combined_main.csv": ({"term", "estimate", "std_error", "conf_low", "conf_high", "p_value", "p_adj_bh"}, {"model", "outcome", "term", "label", "estimate", "std_error", "conf_low", "conf_high", "p_value", "p_adj_bh", "nobs", "n_papers", "r_squared"}),
        "table_ols_poisson_consistency_summary.csv": ({"model", "n_terms_compared", "same_direction_share"}, {"model", "n_terms_compared", "same_direction_n", "same_direction_share", "both_p_lt_05_n", "both_p_lt_05_share", "both_fdr_lt_05_n", "both_fdr_lt_05_share"}),
        "table_chi_square_tests_formatted.csv": ({"chi2", "dof", "cramers_v"}, {"source_file", "group_col", "n_groups", "n_aspects", "chi2", "dof", "p_value_raw", "p_log10", "cramers_v", "chi2_fmt", "p_fmt", "sig", "cramers_v_fmt"}),
        "sample_count__paper_jsd_country_gender_combined.csv": ({"sample", "n_papers", "nobs"}, {"sample", "n_papers", "n_reviews", "nobs"}),
    }
    for name, (expected, allowed) in table_allowlist.items():
        path = PUBLIC_TABLE_DATA / name
        if not path.is_file():
            print(f"SKIP public publication table: {name} is absent")
            continue
        frame = pd.read_csv(path)
        if {"paper_code", "review_id", "review_text", "author_name", "paper_title"}.intersection(frame.columns):
            raise ValueError(f"Non-aggregate public table rejected: {path}")
        if not expected.issubset(frame.columns):
            raise ValueError(f"Public table schema incomplete: {path}")
        extra = set(frame.columns).difference(allowed)
        if extra:
            raise ValueError(f"Unexpected columns in approved public table {path}: {sorted(extra)}")
        save_csv(frame, OUT_TABLE_DIR / name)
        if name == "model_paper_jsd_ols__country_gender_combined_main.csv":
            save_csv(format_regression_table(frame), OUT_TABLE_DIR / "model_paper_jsd_ols__country_gender_combined_main_formatted.csv")
        elif name == "table_ols_poisson_consistency_summary.csv":
            formatted = frame.copy()
            for col in ["same_direction_share", "both_p_lt_05_share", "both_fdr_lt_05_share"]:
                if col in formatted:
                    formatted[col] = formatted[col].map(lambda x: "" if pd.isna(x) else f"{100 * float(x):.2f}%")
            for col in ["n_terms_compared", "same_direction_n", "both_p_lt_05_n", "both_fdr_lt_05_n"]:
                if col in formatted:
                    formatted[col] = formatted[col].map(lambda x: "" if pd.isna(x) else f"{int(x)}")
            save_csv(formatted, OUT_TABLE_DIR / "table_ols_poisson_consistency_summary_formatted.csv")
    for kind in ("country", "gender"):
        name = f"figure_data_jsd_{kind}_coefficients.csv"
        coeff = read_public_aggregate(name, ["estimate", "conf_low", "conf_high", "label"])
        if coeff is None:
            print(f"SKIP {kind} coefficient figure: {name} is absent")
            continue
        coeff = coeff.sort_values("estimate")
        y = np.arange(len(coeff))
        fig, ax = plt.subplots(figsize=(6.8 if kind == "country" else 5.5, max(2.0, 0.35 * len(coeff))), dpi=300)
        ax.errorbar(coeff["estimate"], y, xerr=[coeff["estimate"] - coeff["conf_low"], coeff["conf_high"] - coeff["estimate"]], fmt="o", capsize=3, linewidth=1, markersize=4)
        ax.axvline(0, linestyle="--", linewidth=1)
        ax.set_yticks(y)
        ax.set_yticklabels(coeff["label"])
        ax.set_xlabel("Coefficient on mean pairwise JSD")
        ax.set_ylabel("Country/region (reference: US)" if kind == "country" else "Gender")
        for yi, est in zip(y, coeff["estimate"]):
            ax.text(est, yi, f" {est:.4f}", va="center", fontsize=8)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        fig.tight_layout()
        for suffix in ("png", "pdf"):
            fig.savefig(OUT_FIG_DIR / f"fig_jsd_combined_{kind}_coefficients.{suffix}", bbox_inches="tight")
        plt.close(fig)


## Double-column combined aspect-deviation figure

In [ ]:
country_path = IN_DIR / "figure_data_country_aspect_deviation_pp.csv"
gender_path = IN_DIR / "figure_data_gender_aspect_deviation_pp.csv"
OUT_DIR = OUT_FIG_DIR
if MODE == "public-results":
    country_df = read_public_aggregate(country_path.name, ["group"])
    gender_df = read_public_aggregate(gender_path.name, ["group"])
else:
    country_df = pd.read_csv(country_path) if country_path.is_file() else None
    gender_df = pd.read_csv(gender_path) if gender_path.is_file() else None
if country_df is None or gender_df is None:
    print("SKIP combined aspect-deviation figure: aggregate country or gender matrix is absent.")
else:
    if MODE == "public-results":
        missing_country = set(ANALYSIS_ASPECT_ORDER).difference(country_df.columns)
        missing_gender = set(ANALYSIS_ASPECT_ORDER).difference(gender_df.columns)
        if missing_country or missing_gender:
            raise ValueError(f"Public deviation matrices lack aspects: {sorted(missing_country | missing_gender)}")
    # =========================================================
    # 2. Aspect order and short labels
    # =========================================================
    ASPECT_ORDER = [
        "Methodology",
        "Results",
        "Experiments and Evaluation",
        "Data and Datasets",
        "Figures and Tables",
        "Writing, Language and Terminology",
        "Theory and Models",
        "Comparison and Context",
        "Paper Assessment",
        "Novelty and Impact",
        "Validity, Rigor and Quality",
        "Limitations",
    ]

    ASPECT_SHORT = {
        "Methodology": "Meth.",
        "Results": "Res.",
        "Experiments and Evaluation": "Exp.",
        "Data and Datasets": "Data",
        "Figures and Tables": "Fig.",
        "Writing, Language and Terminology": "Write",
        "Theory and Models": "Theory",
        "Comparison and Context": "Comp.",
        "Paper Assessment": "Assess.",
        "Novelty and Impact": "Novel",
        "Validity, Rigor and Quality": "Valid.",
        "Limitations": "Limit.",
    }

    COUNTRY_ORDER = [
        "US", "CN", "DE", "GB", "JP", "FR", "KR",
        "CA", "CH", "AU", "NL", "ES", "SE"
    ]

    GENDER_ORDER = ["male", "female"]

    # =========================================================
    # 3. Convert to heatmap matrices
    # =========================================================
    country_mat = (
        country_df
        .set_index("group")
        .reindex(COUNTRY_ORDER)
        .reindex(columns=ASPECT_ORDER)
    )

    gender_mat = (
        gender_df
        .set_index("group")
        .reindex(GENDER_ORDER)
        .reindex(columns=ASPECT_ORDER)
    )

    country_mat = country_mat.apply(pd.to_numeric, errors="coerce").fillna(0)
    gender_mat = gender_mat.apply(pd.to_numeric, errors="coerce").fillna(0)

    xlabels = [ASPECT_SHORT[a] for a in ASPECT_ORDER]

    # =========================================================
    # 4. Plot settings
    # =========================================================
    plt.rcParams["font.family"] = "DejaVu Sans"
    plt.rcParams["axes.unicode_minus"] = False
    plt.rcParams["pdf.fonttype"] = 42
    plt.rcParams["ps.fonttype"] = 42

    all_values = np.concatenate([
        country_mat.to_numpy().ravel(),
        gender_mat.to_numpy().ravel()
    ])

    max_abs = np.nanmax(np.abs(all_values))
    max_abs = np.ceil(max_abs * 10) / 10
    if max_abs == 0:
        max_abs = 1.0

    norm = TwoSlopeNorm(vmin=-max_abs, vcenter=0, vmax=max_abs)
    cmap = "RdBu_r"

    # =========================================================
    # 5. Helper function
    # =========================================================
    def draw_heatmap(
        ax,
        mat,
        panel_label,
        show_xlabels=True,
        annot_fmt="{:.1f}",
        annot_fontsize=5.0
    ):
        im = ax.imshow(
            mat.to_numpy(),
            cmap=cmap,
            norm=norm,
            aspect="equal",
            interpolation="nearest"
        )

        n_rows, n_cols = mat.shape

        ax.set_xticks(np.arange(n_cols))
        ax.set_yticks(np.arange(n_rows))

        if show_xlabels:
            ax.set_xticklabels(
                xlabels,
                fontsize=6.5,
                rotation=45,
                ha="right",
                rotation_mode="anchor"
            )
        else:
            ax.set_xticklabels([])

        ax.set_yticklabels(mat.index.tolist(), fontsize=6.8)

        ax.set_xticks(np.arange(-0.5, n_cols, 1), minor=True)
        ax.set_yticks(np.arange(-0.5, n_rows, 1), minor=True)
        ax.grid(which="minor", color="white", linestyle="-", linewidth=0.75)
        ax.tick_params(which="minor", bottom=False, left=False)

        ax.set_ylabel("")

        ax.text(
            -0.02, 1.025,
            panel_label,
            transform=ax.transAxes,
            ha="left",
            va="bottom",
            fontsize=7.8,
            fontweight="bold"
        )

        threshold = max_abs * 0.55
        for i in range(n_rows):
            for j in range(n_cols):
                value = mat.iat[i, j]
                text_color = "white" if abs(value) >= threshold else "black"
                ax.text(
                    j,
                    i,
                    annot_fmt.format(value),
                    ha="center",
                    va="center",
                    fontsize=annot_fontsize,
                    color=text_color
                )

        for spine in ax.spines.values():
            spine.set_linewidth(0.7)

        return im

    # =========================================================
    # 6. Manual layout
    # =========================================================
    # This avoids the large blank space caused by GridSpec + aspect="equal".
    fig_w = 7.2
    fig_h = 8.5
    fig = plt.figure(figsize=(fig_w, fig_h), dpi=300)

    # Main heatmap width in figure coordinates.
    left = 0.18
    heatmap_w = 0.64

    # Convert equal-cell geometry into figure-coordinate heights.
    # axis_height = axis_width * (fig_width / fig_height) * (n_rows / n_cols)
    country_h = heatmap_w * (fig_w / fig_h) * (country_mat.shape[0] / country_mat.shape[1])
    gender_h = heatmap_w * (fig_w / fig_h) * (gender_mat.shape[0] / gender_mat.shape[1])

    # Manually control vertical positions.
    gender_bottom = 0.145
    gap = 0.050
    country_bottom = gender_bottom + gender_h + gap

    ax_gender = fig.add_axes([left, gender_bottom, heatmap_w, gender_h])
    ax_country = fig.add_axes([left, country_bottom, heatmap_w, country_h])

    im = draw_heatmap(
        ax=ax_country,
        mat=country_mat,
        panel_label="(a) First-author country/region",
        show_xlabels=False,
        annot_fmt="{:.1f}",
        annot_fontsize=5.0
    )

    draw_heatmap(
        ax=ax_gender,
        mat=gender_mat,
        panel_label="(b) Inferred first-author gender",
        show_xlabels=True,
        annot_fmt="{:.1f}",
        annot_fontsize=5.0
    )

    #ax_gender.set_xlabel("Review aspect", fontsize=7.5)

    # =========================================================
    # 7. Shorter colorbar
    # =========================================================
    # Align colorbar roughly with the two heatmaps.
    cbar_left = 0.85
    cbar_bottom = gender_bottom + 0.035
    cbar_h = country_h + gender_h + gap - 0.07

    cax = fig.add_axes([cbar_left, cbar_bottom, 0.03, cbar_h])

    cb = fig.colorbar(im, cax=cax)
    cb.set_label("Difference from overall share (pp)", fontsize=7.5)
    cb.ax.tick_params(labelsize=6.5)

    # =========================================================
    # 8. Save
    # =========================================================
    out_png = OUT_DIR / "fig_aspect_deviation_combined_doublecolumn.png"
    out_pdf = OUT_DIR / "fig_aspect_deviation_combined_doublecolumn.pdf"

    fig.savefig(out_png, dpi=300, bbox_inches="tight")
    fig.savefig(out_pdf, bbox_inches="tight")

    plt.close(fig)

    print("Saved:", out_png)
    print("Saved:", out_pdf)


In [ ]:
if MODE == "public-results":
    mapd = read_public_aggregate("descriptive_per_aspect_disagreement_main_strict.csv", ["aspect_label", "mean_mapd"])
else:
    mapd_path = IN_DIR / "descriptive_per_aspect_disagreement_main_strict.csv"
    mapd = pd.read_csv(mapd_path) if mapd_path.is_file() else None
if mapd is None:
    print("SKIP per-aspect disagreement figure: approved aggregate summary is absent")
else:
    plot_mapd = mapd.sort_values("mean_mapd", ascending=True).copy()
    fig, ax = plt.subplots(figsize=(7.0, 5.4), dpi=300)
    ax.barh(plot_mapd["aspect_label"], plot_mapd["mean_mapd"] * 100)
    ax.set_xlabel("Mean pairwise absolute difference (pp)")
    ax.set_ylabel("")
    for i, v in enumerate(plot_mapd["mean_mapd"] * 100):
        ax.text(v, i, f" {v:.2f}", va="center", fontsize=8)
    fig.tight_layout()
    for suffix in ("png", "pdf"):
        fig.savefig(OUT_FIG_DIR / f"fig_per_aspect_disagreement_overall.{suffix}", bbox_inches="tight")
    plt.close(fig)
